# 似然比（LR）检验统计量的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/simulation_lr_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对 Poisson 均值检验构造 LR 统计量，模拟 H0/H1 下的分布并与 χ²(1) 对比。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页并固定种子，为模拟做准备。本 notebook 研究 Poisson 均值检验的**似然比（LR）统计量**。


In [ ]:
%%stata
clear
set more off
set seed 42


**test statistic**

定义统计量计算程序：`lr_test_stat` 生成 n 个 Poisson(λ) 样本，返回 `chi2_stat = 2n[x̄·ln(x̄)-x̄+1]`。
**推导思路**：Poisson(λ) 的对数似然为 -nλ+ln(λ)Σx - Σln(x!)，检验 H0: λ=1 的 LR 统计量 = 2[l(λ̂)-l(1)] = 2n[x̄ln x̄ - x̄ + 1]，正是程序中的公式（含 x! 的项在相减中恰好抵消）。在原假设下它渐近服从 χ²(1)。


In [ ]:
%%stata
cap program drop lr_test_stat
program define lr_test_stat, rclass
	syntax [,obs(integer 100) lambda(real 1)]
	clear
	tempvar x
	quietly{
		set obs `obs'
		gen `x'=rpoisson(`lambda')
		su `x'
		return scalar chi2_stat=2*`obs'*(log(r(mean))*r(mean)-r(mean)+1)
	}
end


**upper tail of chi2(1)**

计算 χ²(1) 的 95% 分位数作为临界值（单侧5%）。


In [ ]:
%%stata
local cv=invchi2tail(1,0.05)


**simulation under H0**

**H0 下的模拟**：默认 λ=1（原假设成立）重复5000次，把统计量直方图与 χ²(1) 密度、临界值竖线叠加；计算实际拒绝率（应接近5%），验证 LR 统计量的渐近分布。


In [ ]:
%%stata
simulate chi2=r(chi2_stat),reps(5000): lr_test_stat
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005


**simulation under H1**

**H1 下的模拟**：以 λ=1.5（备择假设）重复5000次——统计量分布右移、拒绝率显著提高（即功效），与上一节 Wald 检验的模拟思路完全平行。


In [ ]:
%%stata
clear
simulate chi2=r(chi2_stat),reps(5000): lr_test_stat, lambda(1.5)
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005
